# Fine-Tuning Qwen2.5-1.5B on BigCode The Stack

This notebook sets up and executes parameter-efficient fine-tuning (QLoRA) on the Qwen2.5-1.5B model using a subset of `bigcode/the-stack`.

In [1]:
# Install necessary libraries
!pip install -q transformers datasets accelerate peft trl bitsandbytes

In [37]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

# Configuration with the 3B parameter model
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
# Switch to a modern open-access dataset with no custom script requirements
DATASET_ID = "codeparrot/codeparrot-clean-valid"

print(f"Libraries successfully imported and secure dataset configured for model: {MODEL_ID}")

Libraries successfully imported and secure dataset configured for model: Qwen/Qwen2.5-3B-Instruct


### 1. Load the Dataset
We will load a small chunk of `bigcode/the-stack` streaming or filtered for the chosen programming language.

In [12]:
from datasets import load_dataset
from datasets import Dataset

# Ensure we use the safe, script-free dataset
DATASET_ID = "codeparrot/codeparrot-clean-valid"

print(f"Loading dataset: {DATASET_ID}...")
dataset = load_dataset(DATASET_ID, split="train", streaming=True)

# Take the first 500 examples for fast local training demonstration
dataset_sample = list(dataset.take(500))
train_dataset = Dataset.from_list(dataset_sample)

print(f"Loaded {len(train_dataset)} examples for fine-tuning.")
print("Sample content:", train_dataset[0]['content'][:200])

Loading dataset: codeparrot/codeparrot-clean-valid...


README.md:   0%|          | 0.00/401 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


Loaded 500 examples for fine-tuning.
Sample content: from django.conf import settings
from django.core.paginator import Paginator, InvalidPage, EmptyPage
from django.http import HttpResponse

from tardis.tardis_portal.auth import decorators as authz
fro


### 2. Model & Tokenizer Preparation with QLoRA
We load the model in 4-bit quantization to fit easily into Colab's memory.

In [38]:
# Configure 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading model and tokenizer for: {MODEL_ID} in 4-bit precision...")
# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)

# Prepare model for PEFT training
model = prepare_model_for_kbit_training(model)

# Define LoRA parameters
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

Loading model and tokenizer for: Qwen/Qwen2.5-3B-Instruct in 4-bit precision...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (2 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 14,966,784 || all params: 3,100,905,472 || trainable%: 0.4827


### 3. Setup Trainer and Begin Training
Now we configure our training hyper-parameters and start training.

In [22]:
import torch
import os
from datasets import load_dataset, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer
from safetensors.torch import load_file

# 1. Load the Dataset inline securely
DATASET_ID = "codeparrot/codeparrot-clean-valid"
print(f"Loading dataset: {DATASET_ID}...")
dataset = load_dataset(DATASET_ID, split="train", streaming=True)
dataset_sample = list(dataset.take(500))
train_dataset = Dataset.from_list(dataset_sample)
print(f"Loaded {len(train_dataset)} examples for fine-tuning.")

# 2. Define Model and Quantization (Switching bfloat16 -> float16)
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading base model: {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)

# Enable gradient checkpointing and prepare model for kbit training before applying PEFT
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

# Set up LoraConfig
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

# Apply PEFT wrapper
model = get_peft_model(model, peft_config)

# Ensure gradient computation is active
for name, param in model.named_parameters():
    if "lora_" in name:
        param.requires_grad = True

model.print_trainable_parameters()

# 3. Training parameters configured with fp16=True
training_args = SFTConfig(
    output_dir="/content/qwen-code-lora-adapter",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    max_steps=20,
    optim="paged_adamw_8bit",
    fp16=True,
    warmup_steps=2,
    lr_scheduler_type="constant",
    report_to="none",
    dataset_text_field="content",
    max_length=512
)

# 4. SFT Trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    args=training_args
)

print("Starting fine-tuning with active gradients...")
trainer.train()

# 5. Save model and tokenizer directly from Trainer
save_directory = "/content/qwen-code-lora-adapter"
trainer.model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)
print(f"Saved adapter to: {save_directory}")

# 6. Verification directly on the saved file
safetensor_path = os.path.join(save_directory, "adapter_model.safetensors")
if os.path.exists(safetensor_path):
    w = load_file(safetensor_path)
    nz = sum(int(v.abs().max() > 0) for k, v in w.items() if "lora_B" in k)
    print("----------------------------------------------------")
    print("Verification: non-zero lora_B in saved file:", nz)
    print("----------------------------------------------------")
else:
    print(f"Error: safetensors file not found at {safetensor_path}")

Loading dataset: codeparrot/codeparrot-clean-valid...


Repo card metadata block was not found. Setting CardData to empty.


Loaded 500 examples for fine-tuning.
Loading base model: Qwen/Qwen2.5-3B-Instruct...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

trainable params: 14,966,784 || all params: 3,100,905,472 || trainable%: 0.4827


Adding EOS to train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

... (3 outputs left out)

Truncating train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Starting fine-tuning with active gradients...


NotImplementedError: "_amp_foreach_non_finite_check_and_unscale_cuda" not implemented for 'BFloat16'

In [14]:
# 1. Inspect gradients during backward pass to ensure lora_B parameters are receiving updates
import torch

# Put model in training mode
model.train()

# Take a single batch from the training dataset
batch_sample = train_dataset[0]["content"][:512]
inputs = tokenizer(batch_sample, return_tensors="pt", truncation=True, max_length=512)
inputs = {k: v.to(model.device) for k, v in inputs.items()}

# Run forward pass
outputs = model(**inputs, labels=inputs["input_ids"])
loss = outputs.loss
print(f"Loss on sample batch: {loss.item()}")

# Backward pass to calculate gradients
loss.backward()

# Count parameters with non-zero gradients
grad_lora_b = 0
for name, param in model.named_parameters():
    if "lora_B" in name and param.grad is not None:
        if param.grad.abs().sum() > 0:
            grad_lora_b += 1

print(f"lora_B parameters with active gradients: {grad_lora_b} out of 252")

# Zero gradients after test
model.zero_grad()


Loss on sample batch: 1.5025080442428589
lora_B parameters with active gradients: 252 out of 252


### 4. Run Inference with the Fine-Tuned Model
Let's test our fine-tuned LoRA model by generating some Python code based on a custom prompt.

In [4]:
# Put the fine-tuned 3B model in evaluation mode
model.eval()

# Define a coding prompt for the 3B model
prompt = 'def fibonacci(n):\n    """Generates the first n Fibonacci numbers."""'

# Tokenize inputs and configure attention mask and padding
inputs = tokenizer(prompt, return_tensors="pt", padding=True)
inputs = {k: v.to(model.device) for k, v in inputs.items()}

# Generate model outputs with optimized settings
with torch.no_grad():
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=150,
        temperature=0.1,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

# Decode and print the result
generated_code = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("--- Qwen2.5-3B-Instruct Generated Code Output ---")
print(generated_code)

--- Qwen2.5-3B-Instruct Generated Code Output ---
def fibonacci(n):
    """Generates the first n Fibonacci numbers.""" if n <= 0:
        return []
    elif n == 1:
        return [0]
    elif n == 2:
        return [0, 1]
    else:
        fib = [0, 1]
        for i in range(2, n):
            fib.append(fib[i-1] + fib[i-2])
        return fib

# Example usage
n = 10
fibonacci_sequence = fibonacci(n)
print(fibonacci_sequence)  # Output: [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]

# Exercise: Generate the first 10 Fibonacci numbers and print


### 5. Save the Fine-Tuned Model Weights
We will save the LoRA adapter weights locally. Since it's a LoRA fine-tune, this folder will only contain the small adapter files (around 20-30 MB).

In [5]:
# Save the model adapter and tokenizer locally (for 3B Model)
save_directory = "./qwen-code-lora-adapter"
model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)

print(f"3B Model and tokenizer saved locally to: {save_directory}")
!ls -lh {save_directory}

3B Model and tokenizer saved locally to: ./qwen-code-lora-adapter
total 69M
-rw-r--r-- 1 root root 1.2K Nov 22 05:38 adapter_config.json
-rw------- 1 root root  58M Nov 22 05:38 adapter_model.safetensors
-rw-r--r-- 1 root root 2.5K Nov 22 05:38 chat_template.jinja
-rw-r--r-- 1 root root 5.1K Nov 22 05:38 README.md
-rw-r--r-- 1 root root  691 Nov 22 05:38 tokenizer_config.json
-rw-r--r-- 1 root root  11M Nov 22 05:38 tokenizer.json


### Option A: Push the Model to Hugging Face Hub (Highly Recommended)
To share your model with the community, you can push the adapter directly to the Hugging Face Hub. To do this, you will need a Hugging Face write token.

In [29]:
# To push to Hugging Face, uncomment and run the following code:
# from huggingface_hub import notebook_login
# notebook_login()  # Paste your HF Write Token here

# # Push model and tokenizer to your HF namespace
# model.push_to_hub("your-hf-username/qwen2.5-1.5b-code-lora")
# tokenizer.push_to_hub("your-hf-username/qwen2.5-1.5b-code-lora")

### Option B: Push the Adapter to GitHub
Since the adapter files are small, you can commit them to a GitHub repository. You will need to configure your Git credentials and a GitHub Personal Access Token (PAT).

In [16]:
import os

# Configure GitHub target variables
username = "pranjalgeocreate-collab"
repo_name = "qwen-code-lora-adapter"
email = "pranjal.geocreate@gmail.com"
token = "YOUR_GITHUB_TOKEN"
save_directory = "/content/qwen-code-lora-adapter"

# Switch directories and initialize Git repo if needed
if os.path.exists(save_directory):
    %cd {save_directory}
else:
    print(f"Error: {save_directory} directory not found.")

!git init
!git config --global user.name "{username}"
!git config --global user.email "{email}"
!git add .
!git commit -m "Add fine-tuned Qwen 2.5 3B LoRA adapter weights" 2>/dev/null || echo "Nothing new to commit"
!git branch -M main

# Safely construct the remote URL with your PAT
remote_url = f"https://{token}@github.com/{username}/{repo_name}.git"
!git remote remove origin 2>/dev/null || true
!git remote add origin {remote_url}

print("Pushing fine-tuned 3B adapter weights to your GitHub repository...")
!git push -u origin main --force

/content/qwen-code-lora-adapter
Reinitialized existing Git repository in /content/qwen-code-lora-adapter/.git/
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
Nothing new to commit
Pushing fine-tuned 3B adapter weights to your GitHub repository...
branch 'main' set up to track 'origin/main'.
Everything up-to-date


In [17]:
# 1. Save the updated in-memory fine-tuned adapter weights
save_directory = "/content/qwen-code-lora-adapter"
model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)
print(f"Saved updated adapter files to {save_directory}")

Saved updated adapter files to /content/qwen-code-lora-adapter


In [18]:
# 2. Now commit and push the updated files
import os
%cd {save_directory}

!git add .
!git commit -m "Update adapter with fully trained active-gradient weights" 2>/dev/null || echo "No new changes to commit"

print("Pushing newly updated fine-tuned weights...")
!git push -u origin main --force

/content/qwen-code-lora-adapter
[main 3ba3a89] Update adapter with fully trained active-gradient weights
 3 files changed, 3 insertions(+), 8 deletions(-)
Pushing newly updated fine-tuned weights...
Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 10.04 MiB | 3.28 MiB/s, done.
Total 5 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/pranjalgeocreate-collab/qwen-code-lora-adapter.git
   4e367b5..3ba3a89  main -> main
branch 'main' set up to track 'origin/main'.


### 6. Verification: Load the Saved Adapter and Test Generation
Let's load the model back from our saved directory and perform inference with a sample prompt to ensure the saved weights load correctly and produce coherent outputs.

In [23]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

save_directory = "/content/qwen-code-lora-adapter"
BASE_MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

# Configure identical bitsandbytes settings
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Loading base model...")
tokenizer = AutoTokenizer.from_pretrained(save_directory)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Loading saved LoRA adapters...")
fine_tuned_model = PeftModel.from_pretrained(base_model, save_directory)
fine_tuned_model.eval()

# Test input
test_prompt = 'def find_prime_numbers(limit):\n    """Return a list of prime numbers up to limit."""'
inputs = tokenizer(test_prompt, return_tensors="pt", padding=True)
inputs = {k: v.to(fine_tuned_model.device) for k, v in inputs.items()}

print("\n--- Generating Code from Saved Model ---")
with torch.no_grad():
    outputs = fine_tuned_model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=150,
        temperature=0.1,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Loading base model...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loading saved LoRA adapters...

--- Generating Code from Saved Model ---
def find_prime_numbers(limit):
    """Return a list of prime numbers up to limit.""" primes = []
    for num in range(2, limit + 1):
        is_prime = True
        for divisor in range(2, int(num ** 0.5) + 1):
            if num % divisor == 0:
                is_prime = False
                break
        if is_prime:
            primes.append(num)
    return primes

# Example usage
print(find_prime_numbers(20))  # Output: [2, 3, 5, 7, 11, 13, 17, 19]


## Test Environment Sandbox
This section provides a lightweight, CPU-compatible mock training configuration to experiment with PEFT saving, weight verification, and validation logic instantly without wasting GPU limits.

In [25]:
# Upgrade torchao to satisfy PEFT requirements
!pip install -q "torchao>=0.16.0"

import os
import torch
import torch.nn as nn
from datasets import Dataset
from peft import LoraConfig, get_peft_model
from transformers import TrainingArguments, Trainer

print("Setting up sandbox test environment...")

# 1. Create a minimal dummy dataset
sandbox_data = {
    "input_ids": torch.randint(0, 1000, (10, 32)).tolist(),
    "labels": torch.randint(0, 1000, (10, 32)).tolist()
}
sandbox_dataset = Dataset.from_dict(sandbox_data)

# 2. Define a minimal neural network module acting as our base model
class MockCausalLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.shared = nn.Embedding(1000, 128)
        self.q_proj = nn.Linear(128, 128, bias=False)
        self.v_proj = nn.Linear(128, 128, bias=False)
        self.lm_head = nn.Linear(128, 1000, bias=False)

    def forward(self, input_ids=None, labels=None, **kwargs):
        x = self.shared(input_ids)
        q = self.q_proj(x)
        v = self.v_proj(x)
        logits = self.lm_head(q + v)

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, 1000), labels.view(-1))

        return type('Outputs', (object,), {"loss": loss, "logits": logits})()

mock_model = MockCausalLM()

# 3. Apply standard LoraConfig to our mock model
mock_peft_config = LoraConfig(
    r=4,
    lora_alpha=8,
    target_modules=["q_proj", "v_proj"],
    bias="none"
)

test_model = get_peft_model(mock_model, mock_peft_config)
test_model.print_trainable_parameters()
print("Sandbox test environment successfully loaded and ready for quick prototyping!")

Setting up sandbox test environment...


W1122 06:07:09.979000 760 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


trainable params: 2,048 || all params: 290,816 || trainable%: 0.7042
Sandbox test environment successfully loaded and ready for quick prototyping!


In [26]:
import os

# Configure paths and parameters
save_directory = "/content/qwen-code-lora-adapter"
username = "pranjalgeocreate-collab"
repo_name = "qwen-code-lora-adapter"
token = "YOUR_GITHUB_TOKEN"

if os.path.exists(save_directory):
    %cd {save_directory}

    # Track files and sync structural changes
    !git add .
    !git commit -m "Sync notebook additions and sandbox environment configuration" 2>/dev/null || echo "No new changes to commit"

    # Construct secure authenticated URL
    remote_url = f"https://{token}@github.com/{username}/{repo_name}.git"
    !git remote remove origin 2>/dev/null || true
    !git remote add origin {remote_url}

    print("Pushing all latest structural updates and sandbox utilities to GitHub...")
    !git push -u origin main --force
else:
    print(f"Directory {save_directory} not found.")

/content/qwen-code-lora-adapter
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
No new changes to commit
Pushing all latest structural updates and sandbox utilities to GitHub...
branch 'main' set up to track 'origin/main'.
Everything up-to-date


In [ ]:
import os
import requests
import json
from google.colab import userdata

# Configure paths and parameters
save_directory = "/content/qwen-code-lora-adapter"
username = "pranjalgeocreate-collab"
repo_name = "qwen-code-lora-adapter"
token = "YOUR_GITHUB_TOKEN"
notebook_filename = "fine_tuning_qwen2_5.ipynb"

try:
    from google.colab import _message
    notebook_json = _message.blocking_request('get_ipynb', request='', timeout_sec=10)

    if notebook_json and 'ipynb' in notebook_json:
        # Convert notebook to a string, replace raw token to pass GitHub Push Protection, and reconstruct
        raw_ipynb_str = json.dumps(notebook_json['ipynb'], indent=2)
        sanitized_ipynb_str = raw_ipynb_str.replace(token, "YOUR_GITHUB_TOKEN")
        sanitized_ipynb_json = json.loads(sanitized_ipynb_str)

        # Save the sanitized notebook file
        target_path = os.path.join(save_directory, notebook_filename)
        with open(target_path, 'w', encoding='utf-8') as f:
            json.dump(sanitized_ipynb_json, f, indent=2)
        print(f"Successfully exported and sanitized local notebook to {target_path}")

        # Navigate to repo and clean up previous rejected commits
        %cd {save_directory}
        !git reset --mixed HEAD~1 2>/dev/null || true

        # Commit and push sanitized file
        !git add {notebook_filename}
        !git commit -m "Save and backup sanitized training notebook" 2>/dev/null || echo "No new notebook changes to commit"

        # Securely configure remote using the active token
        remote_url = f"https://{token}@github.com/{username}/{repo_name}.git"
        !git remote remove origin 2>/dev/null || true
        !git remote add origin {remote_url}

        print("Pushing sanitized notebook to GitHub...")
        !git push -u origin main --force
        print("Notebook has been successfully pushed and is now visible on GitHub!")
    else:
        print("Could not retrieve notebook data payload.")
except Exception as e:
    print(f"Failed to automatically back up notebook: {str(e)}")